# Create DSL search queries

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from typing import Any

from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [ ]:
docs = Path("..") / "docs"
multi_source_dsl_path = (
    docs / "coresignal" / "Employee APIs" / "Multi-source Employee API.json"
)

with open(multi_source_dsl_path, "r") as file:
    multi_source_dsl_json = file.read()

In [ ]:
class DSLQuery(BaseModel):
    query: dict[str, Any] = Field(description="The DSL search query")

In [ ]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.7-flash",
#     temperature=1.0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
# )

# llm = ChatAnthropic(model="claude-opus-5")  # type: ignore
# dsl_gen = llm.with_structured_output(
#     DSLQuery,
#     method="function_calling",
#     include_raw=True,
# )

llm = ChatOpenAI(model="gpt-5.6-sol")
dsl_gen = llm.with_structured_output(
    DSLQuery,
    method="json_mode",
    # strict=False,
    include_raw=True,
)

In [ ]:
openenings = Path("..") / "openings"
# job_description_path = openenings / "frontend-engineer-berlin.md"
job_description_path = openenings / "technical-founders-associate-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

In [ ]:
prompt = f"""
Generate a DSL search query that finds candidates matching the job description below.

Read the job description carefully and identify which requirements are strict
must-haves (e.g. required skills, required experience, required location/language)
versus nice-to-haves (e.g. skills described as "nice-to-have", "beneficial", "a plus",
or otherwise optional/preferred). Prioritize the query accordingly:
- Encode must-have requirements as `must`/`filter` clauses so only matching candidates
  are returned.
- Encode nice-to-have requirements as `should` clauses (optionally with a `boost`) so
  they influence ranking without excluding otherwise qualified candidates.
Base these priorities only on what is stated in the job description provided, since
different job descriptions will emphasize different requirements.

---

{job_description}

---

# DSL Description:

```json
{multi_source_dsl_json}
```
""".strip()

In [ ]:
messages = [
    ("human", prompt),
]

raw = dsl_gen.invoke(messages)
result = DSLQuery.model_validate(raw["parsed"])

In [ ]:
print(json.dumps(raw["raw"].usage_metadata, indent=2))

In [ ]:
output_dir = Path("..") / "spi" / "search_query" / llm.model
output_dir.mkdir(parents=True, exist_ok=True)
output_file = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + ".json"

with open(output_dir / output_file, "w") as file:
    json.dump(result.query, file, indent=2)

In [ ]:
print(json.dumps(result.query, indent=2))